# Categorization and transaction signals

User-friendly categories, recurring/subscription detection, and anomaly (spike/drift) detection for transactions.

In [ ]:
#| default_exp categorization

In [ ]:
#| export
import pandas as pd
from datetime import datetime, timedelta
from typing import Optional
from collections import defaultdict

In [ ]:
#| export
ESSENTIALS_PRIMARY = {
    "RENT_AND_UTILITIES", "UTILITIES", "HEALTHCARE", "INSURANCE", "LOAN_PAYMENTS",
    "GENERAL_MERCHANDISE", "GENERAL_SERVICES", "GOVERNMENT_AND_NON_PROFIT",
    "RENT", "HOME_IMPROVEMENT", "AUTOMOTIVE_FUEL", "CHILD_SUPPORT", "ALIMONY"
}
DISCRETIONARY_PRIMARY = {
    "ENTERTAINMENT", "RECREATION", "TRAVEL", "PERSONAL_CARE", "SHOPPING",
    "FOOD_AND_DRINK", "TRANSPORTATION"
}
SAVINGS_PRIMARY = {"TRANSFER_IN", "TRANSFER_OUT", "INCOME", "INVESTMENTS", "BANK_FEES"}

ESSENTIALS_DETAILED = {
    "FOOD_AND_DRINK_GROCERIES", "FOOD_AND_DRINK_SUPERMARKETS",
    "RENT_AND_UTILITIES_GAS", "RENT_AND_UTILITIES_ELECTRICITY",
    "RENT_AND_UTILITIES_WATER", "RENT_AND_UTILITIES_INTERNET_AND_CABLE",
    "RENT_AND_UTILITIES_PHONE", "HEALTHCARE_PHARMACY", "HEALTHCARE_PRIMARY",
    "LOAN_PAYMENTS_MORTGAGE_PAYMENT", "LOAN_PAYMENTS_CAR_PAYMENT",
    "INSURANCE_HEALTH", "INSURANCE_AUTO", "INSURANCE_HOME"
}
DISCRETIONARY_DETAILED = {
    "FOOD_AND_DRINK_COFFEE", "FOOD_AND_DRINK_RESTAURANTS", "FOOD_AND_DRINK_FAST_FOOD",
    "FOOD_AND_DRINK_ALCOHOL", "ENTERTAINMENT_MUSIC_AND_AUDIO", "ENTERTAINMENT_SPORTING_EVENTS",
    "ENTERTAINMENT_TV_AND_MOVIES", "RECREATION_GYMS_AND_FITNESS", "RECREATION_HOBBIES",
    "SHOPPING_CLOTHING", "SHOPPING_ELECTRONICS", "TRAVEL_AIRLINES", "TRAVEL_LODGING",
    "TRANSPORTATION_TAXIS_AND_SHARES", "TRANSPORTATION_GAS", "PERSONAL_CARE"
}
SAVINGS_DETAILED = {
    "TRANSFER_IN_DEPOSIT", "TRANSFER_OUT_WITHDRAWAL", "INCOME_WAGES",
    "INVESTMENTS_BUY", "INVESTMENTS_SELL", "BANK_FEES"
}

In [ ]:
#| export
def _map_one(primary: Optional[str], detailed: Optional[str]) -> str:
    """Map Plaid primary/detailed category to user-friendly category."""
    p = (primary or "").strip().upper()
    d = (detailed or "").strip().upper()
    if d and d in SAVINGS_DETAILED:
        return "Savings & Investments"
    if p and p in SAVINGS_PRIMARY:
        return "Savings & Investments"
    if d and d in ESSENTIALS_DETAILED:
        return "Essentials"
    if p and p in ESSENTIALS_PRIMARY:
        return "Essentials"
    if d and d in DISCRETIONARY_DETAILED:
        return "Discretionary"
    if p and p in DISCRETIONARY_PRIMARY:
        return "Discretionary"
    if p:
        return "Discretionary"  # default
    return "Uncategorized"

In [ ]:
#| export
def apply_user_friendly_category(transactions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add column `user_friendly_category` to a transactions DataFrame.
    Expects columns: personal_finance_category (primary), personal_finance_category_detailed (or .detailed).
    """
    df = transactions_df.copy()
    primary = df.get("personal_finance_category", pd.Series(dtype=object))
    if "personal_finance_category_detailed" in df.columns:
        detailed = df["personal_finance_category_detailed"]
    else:
        detailed = df.get("personal_finance_category.detailed", pd.Series(dtype=object))
    df["user_friendly_category"] = [
        _map_one(
            primary.iloc[i] if isinstance(primary, pd.Series) else None,
            detailed.iloc[i] if isinstance(detailed, pd.Series) else None,
        )
        for i in range(len(df))
    ]
    return df

In [ ]:
#| export
def _normalize_merchant(name: Optional[str]) -> str:
    if pd.isna(name) or not name:
        return ""
    return str(name).strip().upper()[:80]


def detect_recurring(transactions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect recurring charges: same merchant + similar amount + same direction at regular intervals.
    Income (amount < 0) and expenses (amount > 0) are tracked separately.
    Adds columns: is_recurring, recurring_type ('income' or 'expense').
    """
    df = transactions_df.copy()
    if "date" not in df.columns and "authorized_date" in df.columns:
        df["_date"] = pd.to_datetime(df["authorized_date"], errors="coerce")
    else:
        df["_date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["_date"]).sort_values("_date")
    df["_merchant"] = df["merchant_name"].map(_normalize_merchant)
    df["_amount_round"] = (df["amount"].abs().round(2)).astype(str)
    df["_sign"] = df["amount"].apply(lambda x: "neg" if x < 0 else "pos")
    df["_key"] = df["_merchant"] + "|" + df["_amount_round"] + "|" + df["_sign"]

    is_recurring = pd.Series(False, index=df.index)
    for key, grp in df.groupby("_key"):
        if grp.shape[0] < 2:
            continue
        dates = grp["_date"].sort_values()
        deltas = dates.diff().dt.days.dropna()
        if len(deltas) == 0:
            continue
        median_days = deltas.median()
        if 25 <= median_days <= 35 or 7 <= median_days <= 9 or 12 <= median_days <= 15:
            is_recurring.loc[grp.index] = True
    df["is_recurring"] = is_recurring
    df["recurring_type"] = df.apply(
        lambda r: ("income" if r["amount"] < 0 else "expense") if r["is_recurring"] else "", axis=1
    )
    df = df.drop(columns=["_date", "_merchant", "_amount_round", "_key", "_sign"], errors="ignore")
    return df

In [ ]:
#| export
def detect_unusual(transactions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag unusual transactions: spikes (amount >> typical) and drift (category spending vs average).
    Adds columns: is_unusual, unusual_reason.
    Expects: amount, and either user_friendly_category or personal_finance_category.
    """
    df = transactions_df.copy()
    cat_col = "user_friendly_category" if "user_friendly_category" in df.columns else "personal_finance_category"
    if cat_col not in df.columns:
        df["is_unusual"] = False
        df["unusual_reason"] = ""
        return df
    df["_abs_amount"] = df["amount"].abs()
    reasons = []
    is_unusual = []
    for idx, row in df.iterrows():
        reason = ""
        cat = row.get(cat_col) or "Uncategorized"
        subset = df[(df[cat_col] == cat) & (df.index != idx)]
        if len(subset) >= 3:
            median_a = subset["_abs_amount"].median()
            p95 = subset["_abs_amount"].quantile(0.95)
            if median_a and row["_abs_amount"] >= 2 * median_a:
                reason = "spike"
            elif row["_abs_amount"] >= p95 and row["_abs_amount"] > median_a * 1.5:
                reason = "spike" if not reason else reason
        is_unusual.append(bool(reason))
        reasons.append(reason)
    df["is_unusual"] = is_unusual
    df["unusual_reason"] = reasons
    df = df.drop(columns=["_abs_amount"], errors="ignore")
    return df

In [ ]:
#| export
def run_enrichment() -> None:
    """
    Load all transactions from DB, compute user_friendly_category, is_recurring, is_unusual,
    and persist those columns back. Requires db_conn, db_sql from jupyter_finance.core.
    Run after transaction refresh; ensures migration 001 has been applied.
    """
    from jupyter_finance.core import db_conn, db_sql

    q = """
    SELECT transaction_id, amount, date, authorized_date, merchant_name,
           personal_finance_category, personal_finance_category_detailed
    FROM transactions
    """
    df = db_sql(q)
    if df.empty:
        return
    df = apply_user_friendly_category(df)
    df = detect_recurring(df)
    df = detect_unusual(df)
    db = db_conn()
    if not db:
        return
    cur = db.cursor()
    for _, row in df.iterrows():
        cur.execute(
            """
            UPDATE transactions
            SET user_friendly_category = %s, is_recurring = %s, is_unusual = %s, unusual_reason = %s
            WHERE transaction_id = %s
            """,
            (
                row.get("user_friendly_category"),
                bool(row.get("is_recurring", False)),
                bool(row.get("is_unusual", False)),
                str(row.get("unusual_reason") or "")[:255],
                row["transaction_id"],
            ),
        )
    db.commit()
    cur.close()
    db.close()

In [ ]:
#| export
def get_transactions_enriched() -> pd.DataFrame:
    """
    Load transactions from DB and return DataFrame with user_friendly_category, is_recurring, is_unusual.
    If DB columns are missing, computes in-memory only (no persistence).
    """
    from jupyter_finance.core import db_sql

    q = """
    SELECT transaction_id, amount, date, authorized_date, merchant_name, name,
           personal_finance_category, personal_finance_category_detailed,
           user_friendly_category, is_recurring, is_unusual, unusual_reason
    FROM transactions
    ORDER BY date DESC
    """
    try:
        df = db_sql(q)
    except Exception:
        q2 = """
        SELECT transaction_id, amount, date, authorized_date, merchant_name, name,
               personal_finance_category, personal_finance_category_detailed
        FROM transactions
        ORDER BY date DESC
        """
        df = db_sql(q2)
        df = apply_user_friendly_category(df)
        df = detect_recurring(df)
        df = detect_unusual(df)
    return df


def get_recurring_summary() -> pd.DataFrame:
    """Return a summary of detected recurring items, separating income and expenses."""
    df = get_transactions_enriched()
    if "is_recurring" not in df.columns or not df["is_recurring"].any():
        rec = df.copy()
        rec = apply_user_friendly_category(rec)
        rec = detect_recurring(rec)
        df = rec
    rec = df[df["is_recurring"] == True].copy()
    if rec.empty:
        return pd.DataFrame(columns=["merchant_name", "typical_amount", "count", "type"])
    rec["_abs"] = rec["amount"].abs()
    if "recurring_type" not in rec.columns:
        rec["recurring_type"] = rec["amount"].apply(lambda x: "income" if x < 0 else "expense")
    summary = (
        rec.groupby([rec["merchant_name"].fillna("Unknown"), "recurring_type"])
        .agg({"transaction_id": "count", "_abs": "median"})
        .rename(columns={"transaction_id": "count", "_abs": "typical_amount"})
        .reset_index()
    )
    summary = summary.rename(columns={"recurring_type": "type"})
    summary["type"] = summary["type"].map({"income": "Income", "expense": "Expense"}).fillna("Expense")
    return summary[["merchant_name", "typical_amount", "count", "type"]].sort_values(
        ["type", "typical_amount"], ascending=[True, False]
    )